# Notebook 1: Document Discovery & Inventory Pipeline
**Objective:** Build an enterprise-grade document discovery and inventory pipeline that scans the raw dataset folders, identifies every file, classifies file types, computes hashes, detects duplicates, and creates a structured inventory of the dataset before OCR or parsing begins.

*Important Note: This notebook focuses solely on discovery, inventory, validation, and reporting. It does NOT perform OCR, text extraction, chunking, or embeddings.*

## Why Document Discovery is Needed in Enterprise AI Systems
Before feeding unstructured data into an AI pipeline (like a Retrieval-Augmented Generation (RAG) system or an Enterprise Knowledge Graph), it is critical to understand the data landscape. 

1. **Data Integrity:** Ensuring no missing or corrupt files enter the pipeline.
2. **Duplicate Prevention:** OCR and Embeddings are computationally expensive. Processing the same document twice wastes resources and biases the AI.
3. **Traceability:** Creating a source-of-truth inventory allows us to map final AI responses back to a specific file hash and timestamp.
4. **Capacity Planning:** Knowing the exact distribution of PDFs vs. Images allows data engineers to allocate appropriate compute resources.

## Theory Section

- **Recursive Directory Traversal:** Systematically walking through a directory tree and all its subdirectories to locate every file, regardless of folder depth.
- **Pathlib:** A modern, object-oriented approach in Python for interacting with the filesystem, replacing older string-based path manipulations (`os.path`).
- **File Hashing with SHA-256:** Generating a unique 256-bit signature for a file's exact binary contents. If a single byte changes, the hash changes completely.
- **MIME Type Detection:** Identifying the true nature of a file (e.g., `application/pdf` or `image/jpeg`) beyond just trusting its file extension.
- **Duplicate Detection:** Using the SHA-256 hashes to find files that have identical contents, even if they have different filenames.
- **File Inventory Creation:** Compiling all gathered metadata (path, size, hash, type) into a structured format (like a CSV or Database) for downstream tracking.
- **Logging and Error Handling:** In enterprise systems, scripts should not silently fail or crash unexpectedly. Logging provides an audit trail of execution.

In [ ]:
import os
import hashlib
import mimetypes
import json
import logging
from pathlib import Path
from datetime import datetime
import pandas as pd

# Configure logging for enterprise-grade execution tracking
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

logger.info("Imports loaded successfully.")

In [ ]:
# Configuration Section
RAW_DATA_DIR = Path("../data/raw")
INVENTORY_DIR = Path("../data/inventory")

# Ensure output directory exists
INVENTORY_DIR.mkdir(parents=True, exist_ok=True)
logger.info(f"Inventory directory configured at: {INVENTORY_DIR.resolve()}")

In [ ]:
def compute_sha256(file_path: Path) -> str:
    """Computes the SHA-256 hash of a file efficiently using chunking."""
    sha256_hash = hashlib.sha256()
    try:
        with open(file_path, "rb") as f:
            # Read and update hash string value in blocks of 4K
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except Exception as e:
        logger.error(f"Failed to hash {file_path.name}: {e}")
        return None

def scan_directory(base_dir: Path) -> list:
    """Recursively scans the directory and gathers file metadata."""
    inventory = []
    
    if not base_dir.exists() or not base_dir.is_dir():
        logger.error(f"Base directory {base_dir} does not exist or is not a directory.")
        return inventory
        
    logger.info(f"Starting recursive scan of {base_dir.resolve()}")
    
    for root, dirs, files in os.walk(base_dir):
        root_path = Path(root)
        for file in files:
            file_path = root_path / file
            
            # Check if file is readable
            is_readable = os.access(file_path, os.R_OK)
            
            # Gather metadata
            file_size = file_path.stat().st_size if is_readable else 0
            mod_time = datetime.fromtimestamp(file_path.stat().st_mtime).isoformat() if is_readable else None
            mime_type, _ = mimetypes.guess_type(file_path)
            file_hash = compute_sha256(file_path) if is_readable else None
            
            inventory.append({
                "Filename": file_path.name,
                "Folder_Path": str(root_path.relative_to(base_dir)),
                "Extension": file_path.suffix.lower(),
                "MIME_Type": mime_type or "unknown",
                "Size_Bytes": file_size,
                "Modified_Time": mod_time,
                "SHA256_Hash": file_hash,
                "Is_Readable": is_readable
            })
            
    logger.info(f"Scan complete. Found {len(inventory)} files.")
    return inventory

# Execute the scan
raw_inventory_data = scan_directory(RAW_DATA_DIR)

In [ ]:
# Convert to DataFrame for easier manipulation
df_inventory = pd.DataFrame(raw_inventory_data)

# Duplicate Detection Code
if not df_inventory.empty:
    # Identify files with the same hash
    # Keep the first occurrence as False (not a duplicate), subsequent ones as True
    df_inventory['Is_Duplicate'] = df_inventory.duplicated(subset=['SHA256_Hash'], keep='first')
    
    duplicate_count = df_inventory['Is_Duplicate'].sum()
    logger.info(f"Duplicate detection complete. Found {duplicate_count} duplicate files.")
else:
    logger.warning("Inventory is empty. Skipping duplicate detection.")
    df_inventory['Is_Duplicate'] = pd.Series(dtype=bool)

df_inventory.head()

In [ ]:
# Folder and Extension Summary Code
if not df_inventory.empty:
    # Count files per folder
    files_per_folder = df_inventory['Folder_Path'].value_counts().to_dict()
    
    # Count files per extension
    files_per_extension = df_inventory['Extension'].value_counts().to_dict()
    
    # Count PDFs vs Images vs Other
    pdf_count = df_inventory[df_inventory['MIME_Type'] == 'application/pdf'].shape[0]
    image_count = df_inventory[df_inventory['MIME_Type'].str.startswith('image/', na=False)].shape[0]
    other_count = len(df_inventory) - (pdf_count + image_count)
    
    category_summary = {
        "PDFs": pdf_count,
        "Images": image_count,
        "Others": other_count
    }
    
    dataset_statistics = {
        "Total_Files": len(df_inventory),
        "Total_Size_MB": round(df_inventory['Size_Bytes'].sum() / (1024 * 1024), 2),
        "Duplicate_Files": int(df_inventory['Is_Duplicate'].sum()),
        "Unreadable_Files": int((~df_inventory['Is_Readable']).sum())
    }
    
    logger.info("Summary statistics generated successfully.")
else:
    files_per_folder = {}
    files_per_extension = {}
    category_summary = {}
    dataset_statistics = {}
    logger.warning("Inventory is empty. Summaries are empty.")

In [ ]:
# Validation Checks
def validate_inventory(df: pd.DataFrame):
    logger.info("Running validation checks...")
    
    if df.empty:
        logger.error("VALIDATION FAILED: Dataset is completely empty.")
        return False
        
    unreadable = df[~df['Is_Readable']]
    if not unreadable.empty:
        logger.warning(f"VALIDATION WARNING: Found {len(unreadable)} unreadable files.")
        
    # Check if expected output paths are writable
    if not os.access(INVENTORY_DIR, os.W_OK):
        logger.error(f"VALIDATION FAILED: Output directory {INVENTORY_DIR} is not writable.")
        return False
        
    logger.info("VALIDATION PASSED: Dataset is ready for inventory export.")
    return True

is_valid = validate_inventory(df_inventory)

In [ ]:
# Save Outputs
if is_valid:
    try:
        # Save complete inventory
        inventory_csv_path = INVENTORY_DIR / "document_inventory.csv"
        df_inventory.to_csv(inventory_csv_path, index=False)
        logger.info(f"Saved document inventory to {inventory_csv_path}")
        
        # Save hashes specifically
        hashes_csv_path = INVENTORY_DIR / "file_hashes.csv"
        df_inventory[['Filename', 'Folder_Path', 'SHA256_Hash', 'Is_Duplicate']].to_csv(hashes_csv_path, index=False)
        logger.info(f"Saved file hashes to {hashes_csv_path}")
        
        # Save folder summary
        folder_summary_path = INVENTORY_DIR / "folder_summary.json"
        with open(folder_summary_path, 'w') as f:
            json.dump({
                "By_Folder": files_per_folder,
                "By_Extension": files_per_extension,
                "By_Category": category_summary
            }, f, indent=4)
        logger.info(f"Saved folder summary to {folder_summary_path}")
        
        # Save overall statistics
        stats_path = INVENTORY_DIR / "dataset_statistics.json"
        with open(stats_path, 'w') as f:
            json.dump(dataset_statistics, f, indent=4)
        logger.info(f"Saved dataset statistics to {stats_path}")
        
    except Exception as e:
        logger.error(f"Failed to save outputs: {e}")

In [ ]:
# Visualization / Summary Display
display(pd.DataFrame(list(dataset_statistics.items()), columns=['Metric', 'Value']))
display(pd.DataFrame(list(category_summary.items()), columns=['Document Type', 'Count']))

## What I Learned
- **The true scale of the raw data:** Identifying exact numbers of PDFs vs. Images helps in planning the OCR and text extraction pipeline (e.g., Tesseract vs. PyPDF).
- **The importance of hashing:** Simply checking filenames is not enough to find duplicates. SHA-256 ensures identical binary content is caught.
- **Enterprise pipeline readiness:** Building a structured inventory *before* doing any AI work prevents debugging nightmares down the line. We now have a source of truth for every file entering our RAG system.

## Interview Readiness: Q&A

**1. Why use recursive directory traversal?**
*Answer:* In enterprise environments, data is rarely flat. Documents are organized in nested hierarchies (e.g., `Department > Equipment > Year`). Recursive traversal (using `os.walk` or `pathlib.rglob`) ensures no files are missed regardless of folder depth.

**2. Why use SHA-256 for hashing?**
*Answer:* SHA-256 is an industry-standard cryptographic hash function. It provides an extremely low probability of collisions (two different files producing the same hash) while being fast enough for large datasets. It acts as a perfect unique fingerprint for file contents.

**3. Why detect duplicates before OCR?**
*Answer:* OCR (Optical Character Recognition) and generating dense vector embeddings are highly computationally expensive operations. Identifying and skipping duplicates *early* saves significant compute time, reduces cloud costs, and prevents the Vector Database from returning duplicate results during semantic search.

**4. Why keep inventory separate from metadata?**
*Answer:* Inventory describes the *physical files* (size, hash, path, readability), while metadata describes the *business context* (Equipment ID, Department, Maintenance Status). Keeping them separate maintains the Single Responsibility Principle. The physical file inventory acts as the foundation upon which business metadata is mapped.

**5. Why validate data before ingestion?**
*Answer:* "Garbage in, garbage out." Validating data (checking for empty folders, unreadable files, or zero-byte files) ensures the pipeline fails fast and cleanly. This prevents downstream crashes in the AI model layers that are much harder to debug.

## Next step: Notebook 2 — PDF Text Extraction and OCR Decisioning